[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/42_stable_diffusion.ipynb)

# 🔴 Hard: Stable Diffusion Sampling Step

Stable Diffusion is a **latent diffusion** model: a VAE maps images to compact 4-channel latents, a text encoder supplies conditioning, and a U-Net predicts noise in latent space. This exercise focuses on the small but central scheduler step after the U-Net has run.

### Before you start

1. A forward diffusion process gradually adds Gaussian noise; `alpha_t` denotes the cumulative signal retention.
2. The U-Net predicts noise twice during classifier-free guidance (CFG): once with the prompt and once without it. CFG combines them as `eps_u + s * (eps_c - eps_u)`.
3. Deterministic DDIM sampling first estimates the clean latent `x0`, then moves from `alpha_t` to `alpha_prev`.

### Signature
```python
def stable_diffusion_step(x_t, eps_cond, eps_uncond, alpha_t, alpha_prev, guidance_scale=7.5):
    # all latent/noise tensors have shape (B, 4, H, W)
    # alpha_t and alpha_prev are scalar cumulative alphas
    # returns x at the previous denoising step
```

Use:

$$\epsilon = \epsilon_u + s(\epsilon_c - \epsilon_u),\qquad \hat{x}_0 = \frac{x_t-\sqrt{1-\alpha_t}\epsilon}{\sqrt{\alpha_t}}$$

then form the deterministic DDIM update using `alpha_prev` and the same `eps`.


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def stable_diffusion_step(x_t, eps_cond, eps_uncond, alpha_t, alpha_prev, guidance_scale=7.5):
    pass  # CFG -> estimate x0 -> deterministic DDIM update


In [ ]:
# 🧪 Debug
torch.manual_seed(0)
x_t = torch.randn(1, 4, 8, 8)
eps_c, eps_u = torch.randn_like(x_t), torch.randn_like(x_t)
a_t, a_prev, scale = torch.tensor(0.5), torch.tensor(0.7), 7.5
out = stable_diffusion_step(x_t, eps_c, eps_u, a_t, a_prev, scale)
# Reference: CFG -> x0 estimate -> deterministic DDIM step
eps = eps_u + scale * (eps_c - eps_u)
x0 = (x_t - torch.sqrt(1 - a_t) * eps) / torch.sqrt(a_t)
ref = torch.sqrt(a_prev) * x0 + torch.sqrt(1 - a_prev) * eps
print('Output shape:', out.shape)
print('Matches DDIM reference:', torch.allclose(out, ref, atol=1e-5))
print('No-CFG output differs:', not torch.allclose(out, stable_diffusion_step(x_t, eps_c, eps_u, a_t, a_prev, 0.0)))


In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('stable_diffusion')
